# E01 · Il registro di chiusura delle aule

Soluzione · Giorno 2

## Dati del turno

In [1]:
righe = [
    ("Aula Blu", "20"),
    ("Aula Verde", "15"),
    ("Aula Blu", "?"),
    ("Aula Rossa", "30"),
    ("Aula Verde", "0"),
    ("Aula Blu", "25"),
    ("Aula Gialla", "121"),
    ("FINE", ""),
    ("Aula Verde", "90"),
]

## A · Leggi e controlla le righe

In [2]:
# La funzione riceve le righe del turno nell’ordine in cui sono arrivate.
def leggi_turno(righe):
    valide = []      # Qui conserviamo le coppie (aula, minuti) accettate.
    scarti = 0       # Conta le righe con una durata non valida.
    esaminate = 0    # Conta tutte le righe controllate, comprese quelle scartate.
    indice = 0       # La prima riga ha posizione 0.

    while indice < len(righe):          # Continua finché c’è una riga da leggere.
        aula, testo = righe[indice]     # Separa il nome dell’aula dalla durata scritta come testo.
        if aula == "FINE":              # Questa riga segnala la chiusura del turno.
            break                       # Esce dal ciclo prima di contare FINE.
        indice += 1                     # Prepara l’indice della prossima riga.

        try:                            # Prova a leggere la durata come numero intero.
            minuti = int(testo)         # Per esempio, "20" diventa 20.
            if minuti < 1:              # Una durata di zero o negativa è fuori intervallo.
                scarti += 1             # Conta questa riga tra gli scarti.
            elif minuti > 120:          # Anche una durata sopra i 120 minuti è fuori intervallo.
                scarti += 1             # Conta questa riga tra gli scarti.
            else:                       # La durata è compresa tra 1 e 120, estremi inclusi.
                valide.append((aula, minuti))  # Aggiunge la coppia in fondo alla lista.
        except ValueError:              # Entra qui se int non può convertire il testo, per esempio "?".
            scarti += 1                 # Conta la durata illeggibile come uno scarto.
        finally:                        # Esegue questo blocco per ogni riga arrivata al try.
            esaminate += 1              # Aggiunge uno al conteggio, sia per una riga valida sia per uno scarto.

    return valide, scarti, esaminate     # Finito il ciclo, restituisce i tre risultati nello stesso ordine.

# Chiama la funzione sui dati del turno e assegna un nome a ciascun risultato.
sessioni, scarti, esaminate = leggi_turno(righe)

## B · Prepara il rapporto per la segreteria

In [3]:
# *sessioni raccoglie le coppie; soglia vale 30 se omessa; **dettagli raccoglie dati come turno="mattina".
def rapporto(*sessioni, soglia=30, **dettagli):
    aule = set()     # Un insieme vuoto per conservare ogni nome di aula una volta sola.
    totali = {}      # Un dizionario vuoto per associare ogni aula al totale dei suoi minuti.

    for aula, minuti in sessioni:           # Legge una coppia valida alla volta.
        aule.add(aula)                      # Registra il nome; l’insieme ignora le ripetizioni.
        if aula in totali:                  # Il dizionario contiene già una chiave con questo nome?
            precedente = totali[aula]       # Sì: recupera i minuti già accumulati per quell’aula.
        else:                               # È la prima sessione di quest’aula.
            precedente = 0                  # Alla prima visita il totale di partenza è zero.
        totali[aula] = precedente + minuti  # Somma i minuti di questa sessione e salva il nuovo totale.

    durate = []                             # Prepara la lista delle singole durate.
    for aula, minuti in sessioni:           # Ripercorre le sessioni nell’ordine originale.
        durate.append(minuti)               # Aggiunge solo i minuti, anche se una durata si ripete.

    sopra_soglia = {}                       # Qui entreranno le aule che raggiungono la soglia.
    for aula, totale in totali.items():     # Legge ogni coppia formata dal nome dell’aula e dal suo totale.
        if totale >= soglia:                # Anche un totale uguale alla soglia è sufficiente.
            sopra_soglia[aula] = totale     # Inserisce nel risultato il nome dell’aula e i suoi minuti.

    return {                                # Restituisce un unico rapporto con cinque campi.
        "dettagli": dettagli,               # I dati del turno passati per nome.
        "aule": aule,                       # I nomi distinti delle aule utilizzate.
        "durate": durate,                   # Le durate delle sessioni, in ordine di arrivo.
        "totali": totali,                   # Tutte le aule con i relativi minuti complessivi.
        "sopra_soglia": sopra_soglia,       # Le sole aule che raggiungono la soglia scelta.
    }                                       # Chiude il dizionario restituito dalla funzione.

## C · Verifica il risultato

In [4]:
# L’asterisco passa alla funzione le singole coppie della lista. La soglia predefinita è 30.
base = rapporto(*sessioni, turno="mattina")

# Usa le stesse sessioni e lo stesso turno, cambiando soltanto la soglia.
alternativo = rapporto(*sessioni, soglia=40, turno="mattina")

# Mostra quante righe sono state esaminate e quante sono state scartate.
print("Righe esaminate", esaminate, "– Scarti", scarti)
print(base)  # Mostra tutti e cinque i campi del primo rapporto.

# Il confronto dà True se ogni riga esaminata è stata conteggiata come valida oppure scartata.
print("Conteggio corretto", len(sessioni) + scarti == esaminate)
print("Soglia 30", base["sopra_soglia"])         # Blu (45) e Rossa (30) raggiungono la prima soglia.
print("Soglia 40", alternativo["sopra_soglia"])  # Solo Blu raggiunge anche la seconda soglia.

# FINE è la prima riga: la funzione restituisce ([], 0, 0) e ignora la successiva sessione di Blu.
print("Turno chiuso subito", leggi_turno([("FINE", ""), ("Aula Blu", "20")]))

# Estensione facoltativa: ogni coppia contiene il nome dell’aula in posizione 0 e il totale in posizione 1.
# sorted usa il totale come criterio e ordina le coppie dal numero di minuti più piccolo al più grande.
classifica = sorted(base["totali"].items(), key=lambda coppia: coppia[1])
print("Aule in ordine di durata", classifica)  # Mostra Verde (15), Rossa (30) e Blu (45).

Righe esaminate 7 – Scarti 3
{'dettagli': {'turno': 'mattina'}, 'aule': {'Aula Blu', 'Aula Verde', 'Aula Rossa'}, 'durate': [20, 15, 30, 25], 'totali': {'Aula Blu': 45, 'Aula Verde': 15, 'Aula Rossa': 30}, 'sopra_soglia': {'Aula Blu': 45, 'Aula Rossa': 30}}
Conteggio corretto True
Soglia 30 {'Aula Blu': 45, 'Aula Rossa': 30}
Soglia 40 {'Aula Blu': 45}
Turno chiuso subito ([], 0, 0)
Aule in ordine di durata [('Aula Verde', 15), ('Aula Rossa', 30), ('Aula Blu', 45)]
